# Beyond BLAST

In [1]:
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = 0;
using Pkg
Pkg.activate("blast_code"; io=devnull)
Pkg.resolve(; io=devnull)
Pkg.instantiate(; io=devnull)

using Base.Threads
using NPZ
using DataInterpolations
using Interpolations
using FastChebInterp
using BenchmarkTools
using FFTW
using FastTransforms
using Dates
using TOML
using Plots
using Plots.Measures
using QuadGK
using LaTeXStrings
using Tullio
using StaticArrays
using LoopVectorization
using LinearAlgebra
using Unitful
using Revise
using SpecialFunctions
using DifferentialEquations
using Cosmology
using NumericalIntegration
using CSV
using DataFrames
using JSON

[ Info: Precompiling DataInterpolations [82cc6244-b520-54b8-b5a6-8a565e85f1d0]
Precompiling packages...
   2065.6 ms  ✓ DataInterpolations → DataInterpolationsChainRulesCoreExt
  1 dependency successfully precompiled in 3 seconds. 37 already precompiled.
[ Info: Precompiling DataInterpolationsChainRulesCoreExt [187dfacc-d000-5da0-94aa-5ec8eb045cfd]
┌ Warning: Module DataInterpolations with build ID fafbfcfd-92c2-15c2-ca93-3af5803f4a37 is missing from the cache.
│ This may mean DataInterpolations [82cc6244-b520-54b8-b5a6-8a565e85f1d0] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:2022
[ Info: Skipping precompilation since __precompile__(false). Importing DataInterpolationsChainRulesCoreExt [187dfacc-d000-5da0-94aa-5ec8eb045cfd].
Precompiling packages...
    570.4 ms  ✓ FastChebInterp
  1 dependency successfully precompiled in 1 seconds. 18 already precompiled.
[ Info: Precompiling FastChebInterp [cf66c380-9a80-432c-aff8-4f9c79c0bdde]
Precompi

In [334]:
include("blast_code/src/Blast.jl")
include("blast_code/src/blast_tutorials.jl")
using .Blast
using .blast_tutorials;
include("blast_code/src/galaxy_galaxy.jl")
#include("blast_code/src/galaxy_shear.jl")
include("blast_code/src/shear_shear.jl")
using .galaxy_galaxy
#using .galaxy_shear
using .shear_shear

In [335]:
timestamp = Dates.format(now(), "yyyy_mm_dd_HHMMSS")
output_dir = "out/runs/run_$timestamp"
plot_subdir = joinpath(output_dir, "plots")
Sl_plots = joinpath(plot_subdir, "Sl_plots")
Kernel_plots = joinpath(plot_subdir, "kernels")
quantity_subdir = joinpath(output_dir, "quantities")
chebcoefs = joinpath(quantity_subdir, "chebcoefs")
Sl = joinpath(quantity_subdir, "Sl")
mkpath(output_dir)
mkpath(plot_subdir)
mkpath(quantity_subdir)
mkpath(chebcoefs)
mkpath(Sl)
mkpath(Sl_plots)
mkpath(Kernel_plots)
println("Folders in place: ", output_dir, ", ", plot_subdir, " and ", quantity_subdir)

Folders in place: out/runs/run_2026_06_22_143314, out/runs/run_2026_06_22_143314/plots and out/runs/run_2026_06_22_143314/quantities


### Getting background quantities:

redshift $z$, comoving distance $\chi(z)$

In [336]:
#Background quantities
z_b = npzread("blast_code/data/background/z.npy") # array 
χ_b = npzread("blast_code/data/background/chi.npy") # array 
# using Akima interpolation
z_of_χ = DataInterpolations.AkimaInterpolation(z_b, χ_b); # z(χ)
chi_of_z = DataInterpolations.AkimaInterpolation(χ_b, z_b); # χ(z)

#### Setting parameters

In [337]:
cosmo = Blast.FlatΛCDM()
n_chi = 96                                            # Number of comoving distance points
x_range = LinRange(26, 7000, n_chi)                   # in Mpc/h: from 26 Mpc/h to 7000 Mpc/h
z_range = z_of_χ.(x_range)
#ℓ = Blast.ℓ
ℓ = LinRange(2, 200, 100)
x_min = minimum(x_range)                              # minimum comoving distance [Mpc/h]
x_max = maximum(x_range)                              # maximum comoving distance [Mpc/h]
chi = LinRange(x_min, x_max, n_chi)                   # array of comoving distances [Mpc/h]
zed = z_of_χ.(chi)                                    # array of redshifts corresponding to the comoving distances
n_z = length(zed)
n5k_bins = npzread("blast_code/data/dNdzs_fullwidth.npz")
z_n5k = n5k_bins["z_cl"]
zmin = minimum(z_n5k)                                 # minimum redshift
zmax = maximum(z_n5k)                                 # maximum redshift
kmax = 200/13                                         # maximum wavenumber (small scales) [h/Mpc]
kmin = 2.5/x_max                                      # minimum wavenumber (large scales) [h/Mpc]
n_cheb = 119                                          # number of Chebyshev nodes 
β = 2                                                 # exponent depending on the probe: 0 for Galaxy - Shear, 2 for Galaxy - Galaxy, -2 for Shear - Shear
k_cheb = chebpoints(n_cheb, log10(kmin), log10(kmax)) # number of Chebyshev "points"
N = 2^15+1;

In [ ]:
# ─── Brute-force setup ────────────────────────────────────────────────────────
# We reuse variables already in scope from the cells above:
#   ℓ, z_range, zmin, zmax, kmin, kmax, n_cheb, N, chi_of_z, cheb_coeff_gal
# No new includes needed.

using SpecialFunctions: sphericalbesselj
using NPZ

println("Brute-force setup:")
println("  n_ell    = ", length(ℓ))
println("  n_chi    = ", length(z_range))
println("  n_cheb   = ", n_cheb)
println("  N (k pts)= ", N)


In [ ]:
# ─── Brute-force W_tilde ─────────────────────────────────────────────────────
#
# W_tilde_bf[ℓ_idx, i, p, l]  (1-indexed, same layout as W_tilde)
#
# Formula:
#   W̃ᵢₚₗ^(ℓ) = Σₖ  wₖ · Tₗ(zₖ) · j_ℓ(χᵢ·kₖ) · j_ℓ(χₚ·kₖ)
#
# where:
#   k_grid  = Clenshaw-Curtis nodes on [kmin, kmax]
#   w       = Clenshaw-Curtis weights on [kmin, kmax]
#   T_l(zₖ) = Chebyshev polynomial of degree l-1 evaluated at the MAPPED k-node
#             (mapping: xₖ = (2·kₖ - (kmax+kmin))/(kmax-kmin) ∈ [-1,1])
#   χᵢ      = chi_of_z(z_range[i])

k_grid_bf = Blast.get_clencurt_grid_z(kmin, kmax, N)     # CC nodes on [kmin,kmax]
w_bf      = Blast.get_clencurt_weights_z(kmin, kmax, N)  # CC weights

Nz = length(z_range)
chi_vals_bf = chi_of_z.(z_range)                         # χ(zᵢ), length Nz

# Chebyshev polynomials T_l on k-grid (same recurrence as new_funcs.jl)
# x_k: map k_grid -> [-1,1]
x_k = @. (2 * k_grid_bf - (kmax + kmin)) / (kmax - kmin)
T_bf = zeros(n_cheb, N)          # T_bf[l, k]  (l = 1…n_cheb, i.e. degree 0…n_cheb-1)
T_bf[1, :] .= 1.0
if n_cheb >= 2
    T_bf[2, :] .= x_k
end
for l in 3:n_cheb
    @. T_bf[l, :] = 2 * x_k * T_bf[l-1, :] - T_bf[l-2, :]
end

n_ell_bf = length(ℓ)
W_tilde_bf = zeros(Float64, n_ell_bf, Nz, Nz, n_cheb)

println("Computing W_tilde_bf: $(n_ell_bf) ℓ-values × $(Nz)×$(Nz) chi pairs × $(n_cheb) Cheb modes × $N k-points")
println("(This may take several minutes…)")

t_start = time()
for (li, ell) in enumerate(ℓ)
    # Pre-compute Bessel matrix: Bessel_bf[i, k] = j_ell(χᵢ · kₖ)
    Bessel_bf = zeros(Nz, N)
    for i in 1:Nz
        for kk in 1:N
            @inbounds Bessel_bf[i, kk] = sphericalbesselj(ell, chi_vals_bf[i] * k_grid_bf[kk])
        end
    end

    # Quadruple loop: i, p, l, k
    for p in 1:Nz
        for i in 1:Nz
            for l in 1:n_cheb
                acc = 0.0
                for kk in 1:N
                    @inbounds acc += w_bf[kk] * T_bf[l, kk] * Bessel_bf[i, kk] * Bessel_bf[p, kk]
                end
                @inbounds W_tilde_bf[li, i, p, l] = acc
            end
        end
    end

    if li % 10 == 0
        elapsed = time() - t_start
        println("  ℓ step $li / $n_ell_bf  ($(round(elapsed, digits=1))s elapsed)")
    end
end

elapsed_total = time() - t_start
println("\nDone. W_tilde_bf computed in $(round(elapsed_total, digits=1))s")
println("Size of W_tilde_bf: ", size(W_tilde_bf))
npzwrite(joinpath(output_dir, "W_tilde_bf.npy"), W_tilde_bf)
println("Saved to: ", joinpath(output_dir, "W_tilde_bf.npy"))


In [ ]:
# ─── Brute-force C_ell^{gg} ──────────────────────────────────────────────────
#
# Formula:
#   C_ℓ^{gg} = Σₘ Σₙ cₘ·cₙ · Σᵢ Σⱼ W̃ᵢⱼₘ^(ℓ) · W̃ᵢⱼₙ^(ℓ)
#
# where cₘ = cheb_coeff_gal[m]  (length n_cheb)

C_ell_gal_bf = zeros(Float64, n_ell_bf)

println("Computing C_ell_gal_bf brute-force…")
t2 = time()

for (li, ell) in enumerate(ℓ)
    acc_ell = 0.0
    for m in 1:n_cheb
        for n in 1:n_cheb
            cm = cheb_coeff_gal[m]
            cn = cheb_coeff_gal[n]
            mn_factor = cm * cn
            for i in 1:Nz
                for j in 1:Nz
                    @inbounds acc_ell += mn_factor * W_tilde_bf[li, i, j, m] * W_tilde_bf[li, i, j, n]
                end
            end
        end
    end
    C_ell_gal_bf[li] = acc_ell
end

elapsed2 = time() - t2
println("Done in $(round(elapsed2, digits=1))s")
npzwrite(joinpath(output_dir, "C_ell_gal_bf.npy"), C_ell_gal_bf)
println("Saved to: ", joinpath(output_dir, "C_ell_gal_bf.npy"))


In [ ]:
# ─── Plot: brute-force vs optimized ──────────────────────────────────────────
using Plots, LaTeXStrings

p = plot(ℓ, ℓ .* (ℓ .+ 1) .* C_ell_gal_bf,
         label  = "Brute force",
         xlabel = L"\ell",
         ylabel = L"\ell(\ell+1)\,C_\ell^{gg}",
         title  = "Galaxy–Galaxy Angular Power Spectrum (Brute Force)",
         xscale = :log10,
         yscale = :log10,
         lw     = 2,
         size   = (800, 500),
         dpi    = 200)

savefig(p, joinpath(output_dir, "plots/C_ell_gal_bf.png"))
println("Plot saved to: ", joinpath(output_dir, "plots/C_ell_gal_bf.png"))
display(p)
